# Fase 5 — Simulación Monte Carlo del Mundial 2026

Simula 10.000 veces el torneo completo usando las probabilidades del modelo de Logistic Regression (Fase 4).  
Para cada selección se calcula la probabilidad de superar cada ronda:
> Clasificar desde grupos → Octavos → Cuartos → Semis → Final → **Campeón**

**Formato del torneo:**  
48 equipos · 12 grupos · Top 2 + 8 mejores 3os → 32 clasificados · Ronda de 32 → R16 → QF → SF → Final

**Nota sobre el bracket:** Se usa un bracket aproximado basado en pods de grupos (A-C, D-F, G-I, J-L)  
que garantiza que equipos del mismo grupo no se crucen hasta Cuartos. El bracket oficial FIFA puede diferir en algunos emparejamientos.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.notebook import tqdm

# Añadir src al path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / "src"))

from simulation import (
    load_data,
    build_team_stats,
    build_prob_cache,
    simulate_tournament,
    save_results,
    GROUPS,
    ROUNDS,
)

# Estilo de gráficos
plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
PALETTE = sns.color_palette("husl", 6)

print("Librerías cargadas ✓")

## 1. Carga de datos y pre-computación de probabilidades

In [ ]:
groups_df, predictions_df, features_df, model = load_data()
team_stats = build_team_stats(features_df, groups_df)
teams = groups_df["team"].tolist()

print(f"Equipos: {len(teams)} | Partidos de fase de grupos: {len(predictions_df)}")
print("Construyendo cache de probabilidades para eliminatorias...")
prob_cache = build_prob_cache(teams, team_stats, model)
print(f"Cache: {len(prob_cache):,} pares de equipos ✓")

## 2. Monte Carlo — 10.000 simulaciones

In [ ]:
N_SIMULATIONS = 10_000
SEED = 42

rng = np.random.default_rng(SEED)

# Acumuladores
counts = {t: {r: 0 for r in ROUNDS} for t in teams}

for _ in tqdm(range(N_SIMULATIONS), desc="Simulando torneos", unit="sim"):
    result, _ = simulate_tournament(groups_df, predictions_df, team_stats, prob_cache, rng)
    for team, best_round in result.items():
        best_idx = ROUNDS.index(best_round)
        for r in ROUNDS[: best_idx + 1]:
            counts[team][r] += 1

print(f"\n{N_SIMULATIONS:,} simulaciones completadas ✓")

In [ ]:
# Construir DataFrame de resultados
rows = []
for team in teams:
    rows.append({
        "team": team,
        "group": groups_df.loc[groups_df["team"] == team, "group"].iloc[0],
        "confederation": groups_df.loc[groups_df["team"] == team, "confederation"].iloc[0],
        "qualify":       counts[team]["round_of_32"] / N_SIMULATIONS,
        "round_of_16":   counts[team]["round_of_16"] / N_SIMULATIONS,
        "quarter_final": counts[team]["quarter_final"] / N_SIMULATIONS,
        "semi_final":    counts[team]["semi_final"] / N_SIMULATIONS,
        "final":         counts[team]["final"] / N_SIMULATIONS,
        "champion":      counts[team]["champion"] / N_SIMULATIONS,
    })

sim_df = (
    pd.DataFrame(rows)
    .sort_values("champion", ascending=False)
    .reset_index(drop=True)
)
sim_df.head(10)

## 3. Top 16 favoritos — Probabilidades acumuladas por ronda

In [ ]:
top16 = sim_df.head(16).copy()

# Formatear como porcentajes para visualización
fmt_cols = ["qualify", "round_of_16", "quarter_final", "semi_final", "final", "champion"]
display_df = top16[["team", "group", "confederation"] + fmt_cols].copy()
for c in fmt_cols:
    display_df[c] = (display_df[c] * 100).round(1).astype(str) + "%"

display_df.columns = ["Equipo", "Grupo", "Conf.", "Clasifica", "R16", "QF", "SF", "Final", "Campeón"]
display_df = display_df.reset_index(drop=True)
display_df.index += 1
display_df

## 4. Visualización — Probabilidad de ser campeón (Top 20)

In [ ]:
top20 = sim_df.head(20).copy()

# Mapa de colores por confederación
CONF_COLORS = {
    "UEFA": "#2166AC",
    "CONMEBOL": "#006837",
    "CONCACAF": "#D73027",
    "AFC": "#F46D43",
    "CAF": "#FEE090",
    "OFC": "#A6CEE3",
}
colors = [CONF_COLORS.get(c, "#999999") for c in top20["confederation"]]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(
    top20["team"][::-1],
    top20["champion"][::-1] * 100,
    color=colors[::-1],
    edgecolor="white",
    linewidth=0.5,
)

# Etiquetas de valor
for bar, val in zip(bars, top20["champion"][::-1] * 100):
    ax.text(val + 0.15, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", fontsize=8.5)

ax.set_xlabel("Probabilidad de ser campeón (%)", fontsize=10)
ax.set_title(
    f"Favoritos al título — Mundial 2026\n"
    f"Simulación Monte Carlo ({N_SIMULATIONS:,} iteraciones, Logistic Regression)",
    fontsize=11, pad=12,
)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))

# Leyenda de confederaciones
from matplotlib.patches import Patch
legend_elements = [Patch(fc=v, label=k) for k, v in CONF_COLORS.items()]
ax.legend(handles=legend_elements, loc="lower right", fontsize=8, framealpha=0.7)

plt.tight_layout()
fig.savefig(ROOT / "reports" / "champion_probabilities.png", bbox_inches="tight")
plt.show()
print("Guardado: reports/champion_probabilities.png")

## 5. Mapa de calor — Probabilidades por ronda (Top 20)

In [ ]:
heat_cols = ["qualify", "round_of_16", "quarter_final", "semi_final", "final", "champion"]
heat_labels = ["Clasifica", "R16", "QF", "SF", "Final", "Campeón"]

heat_data = top20.set_index("team")[heat_cols] * 100

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    heat_data,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.4,
    ax=ax,
    xticklabels=heat_labels,
    cbar_kws={"label": "Probabilidad (%)"},
)
ax.set_title(
    f"Probabilidades por ronda — Top 20 favoritos\n"
    f"Mundial 2026 · {N_SIMULATIONS:,} simulaciones",
    fontsize=11, pad=10,
)
ax.set_ylabel("")
ax.set_xlabel("")
plt.tight_layout()
fig.savefig(ROOT / "reports" / "round_probabilities_heatmap.png", bbox_inches="tight")
plt.show()
print("Guardado: reports/round_probabilities_heatmap.png")

## 6. Bracket más probable

Para cada posición del bracket, el equipo que aparece con más frecuencia en esa posición.

In [ ]:
# Para el bracket más probable, corremos 1 simulación deterministamente
# usando la mediana de probabilidades (prob más alta en cada partido)
# → equivale a simular con el equipo más probable ganando cada partido.

from collections import defaultdict
from simulation import _resolve_r32_bracket, _best_thirds, _simulate_group, R32_TEMPLATE

# Acumular quién llega a cada ronda de eliminatoria (para el bracket más probable)
N_BRACKET = 10_000  # ya calculado arriba, usamos counts directamente

# Los mismos counts ya calculados: sim_df tiene las probs
print("Bracket más probable (equipo con mayor probabilidad en cada ronda):")
print()

bracket_cols = {
    "Ronda de 32 (Clasifican)": "qualify",
    "Octavos (R16)": "round_of_16",
    "Cuartos (QF)": "quarter_final",
    "Semifinales (SF)": "semi_final",
    "Final": "final",
    "Campeón": "champion",
}

for label, col in bracket_cols.items():
    top_n = 4 if col in ["qualify"] else (8 if col == "round_of_16" else 
             4 if col == "quarter_final" else 2 if col == "semi_final" else 1)
    # Ajustar número por ronda
    n_teams = {
        "qualify": 32, "round_of_16": 16, "quarter_final": 8,
        "semi_final": 4, "final": 2, "champion": 1
    }[col]
    top = sim_df.nlargest(n_teams, col)[["team", "group", col]].copy()
    top[col] = (top[col] * 100).round(1).astype(str) + "%"
    top.columns = ["Equipo", "Grupo", "Prob."]
    top = top.reset_index(drop=True)
    top.index += 1
    print(f"{'─'*50}")
    print(f"  {label} — Top {n_teams}")
    print(top.to_string())
    print()

## 7. Probabilidades por confederación

In [ ]:
conf_summary = (
    sim_df.groupby("confederation")[["qualify", "round_of_16", "quarter_final",
                                     "semi_final", "final", "champion"]]
    .sum()
    .sort_values("champion", ascending=False)
)
conf_pct = (conf_summary * 100).round(1)
conf_pct.columns = ["Clasifica", "R16", "QF", "SF", "Final", "Campeón"]
print("Suma de probabilidades por confederación (%, suma de equipos):")
conf_pct

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
conf_pct[["Clasifica", "R16", "QF", "SF", "Final", "Campeón"]].plot(
    kind="bar", ax=ax, width=0.75, colormap="tab10"
)
ax.set_title("Suma de probabilidades por confederación\nMundial 2026", fontsize=11)
ax.set_xlabel("")
ax.set_ylabel("Probabilidad acumulada (%)")
ax.legend(loc="upper right", fontsize=8)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
fig.savefig(ROOT / "reports" / "confederation_probabilities.png", bbox_inches="tight")
plt.show()
print("Guardado: reports/confederation_probabilities.png")

## 8. Tabla completa de los 48 equipos

In [ ]:
pd.set_option("display.max_rows", 50)
full_table = sim_df.copy()
for c in ["qualify", "round_of_16", "quarter_final", "semi_final", "final", "champion"]:
    full_table[c] = (full_table[c] * 100).round(1)
full_table.columns = ["Equipo", "Grupo", "Conf.", "Clasifica%", "R16%", "QF%", "SF%", "Final%", "Campeón%"]
full_table.index = range(1, 49)
full_table

## 9. Guardar resultados

In [ ]:
out_path = save_results(sim_df)
print(f"Resultados guardados en: {out_path}")
print(f"Shape: {sim_df.shape}")
sim_df.head(5)

---
## Resumen de la Fase 5

| Parámetro | Valor |
|---|---|
| Simulaciones | 10.000 |
| Modelo | Logistic Regression (log-loss test: 0.8454) |
| Fase de grupos | Probs. pre-calculadas con H2H histórico |
| Eliminatorias | Cache batch de 2.256 pares de equipos |
| Goles (tiebreaker) | Simulación Poisson ajustada por ELO |
| Bracket | Pods interleaved (aproximado al oficial FIFA) |
| Output | `reports/simulation_results.csv` |

**Siguiente fase:** Fase 6 — Dashboard (Power BI / Tableau)